# S03E04 — Negotiations (Tool Server)

Agent centrali szuka miast sprzedających podzespoły do turbiny wiatrowej.
Udostępniamy mu jedno narzędzie `/api/search` które:
1. Przyjmuje opis przedmiotu w języku naturalnym
2. Dopasowuje go do katalogu (keyword + LLM)
3. Zwraca listę miast sprzedających ten przedmiot

**Wymagania**: serwer Flask + publiczny tunel (serveo.net)

In [ ]:
import os, requests, time, subprocess, re
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.getenv("AI_DEVS_API_KEY")
VERIFY_URL = "https://hub.ag3nts.org/verify"

## 1. Pobierz dane CSV

In [ ]:
BASE = "https://hub.ag3nts.org/dane/s03e04_csv/"
for fname in ["cities.csv", "connections.csv", "items.csv"]:
    path = os.path.join(os.getcwd(), fname)
    if not os.path.exists(path):
        r = requests.get(BASE + fname)
        with open(path, "wb") as f:
            f.write(r.content)
        print(f"Downloaded {fname}")
    else:
        print(f"{fname} already exists")

## 2. Uruchom serwer Flask + tunel serveo

Serwer (`server.py`) udostępnia endpoint `/api/search` z logiką:
- keyword pre-filter po katalogu items.csv
- Claude Haiku do disambiguacji gdy wiele wyników
- lookup miast z connections.csv

In [ ]:
# Uruchom Flask server w tle
env = os.environ.copy()
env["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY", "")

server_proc = subprocess.Popen(
    ["python3", "server.py"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)

# Sprawdz czy serwer dziala
try:
    r = requests.post("http://localhost:5001/api/search",
                      json={"params": "turbina wiatrowa 48V"})
    print(f"Server OK: {r.json()['output'][:80]}")
except Exception as e:
    print(f"Server error: {e}")

In [ ]:
# Uruchom tunel SSH do serveo.net
tunnel_proc = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=60",
     "-R", "80:localhost:5001", "serveo.net"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)

# Odczytaj URL tunelu
output = tunnel_proc.stdout.readline().decode().strip()
print(f"Tunnel output: {output}")

tunnel_url = re.search(r'(https://[^\s]+)', output)
if tunnel_url:
    TUNNEL_URL = tunnel_url.group(1)
    print(f"Tunnel URL: {TUNNEL_URL}")
else:
    # Fallback - odczytaj kolejna linie
    output2 = tunnel_proc.stdout.readline().decode().strip()
    tunnel_url = re.search(r'(https://[^\s]+)', output2)
    TUNNEL_URL = tunnel_url.group(1) if tunnel_url else "ERROR"
    print(f"Tunnel URL: {TUNNEL_URL}")

In [ ]:
# Test tunelu
r = requests.post(f"{TUNNEL_URL}/api/search",
                  json={"params": "turbina wiatrowa 48V"})
print(f"Tunnel test: {r.json()}")

## 3. Zarejestruj narzędzie w centrali

In [ ]:
resp = requests.post(VERIFY_URL, json={
    "apikey": API_KEY,
    "task": "negotiations",
    "answer": {
        "tools": [
            {
                "URL": f"{TUNNEL_URL}/api/search",
                "description": (
                    "Search for items available in cities. "
                    "Send the item name or description in Polish or English "
                    "as the params field (e.g. 'turbina wiatrowa 48V' or 'akumulator 48V 150Ah'). "
                    "Returns the item name and list of cities where it can be purchased. "
                    "Use this tool to find which cities sell a specific item."
                ),
            }
        ]
    },
})
print(f"Registration: {resp.json()}")

## 4. Czekaj na wynik i sprawdź flagę

In [ ]:
# Poczekaj ~45 sekund na agenta, potem sprawdz wynik
print("Czekam 45s na agenta...")
time.sleep(45)

resp = requests.post(VERIFY_URL, json={
    "apikey": API_KEY,
    "task": "negotiations",
    "answer": {"action": "check"},
})
result = resp.json()
print(f"Result: {result}")

if result.get("code") != 0:
    print("Jeszcze nie gotowe, sprobuj ponownie za chwile...")

In [ ]:
# Cleanup - zatrzymaj serwer i tunel
server_proc.terminate()
tunnel_proc.terminate()
print("Serwer i tunel zatrzymane.")